# Predictive Modeling

This notebook uses `data/processed/stock_features.csv` to compare baseline and supervised classification models for next-day stock direction prediction.

The prediction task is: **predict whether the next trading day's return is positive**. This is framed as a binary classification problem where the positive class means the next trading day closed higher than the current trading day.

> **Limitations note:** Historical price and volume features alone are not enough to guarantee profitable trading performance. The models below evaluate directional classification on historical data only; they do not include transaction costs, slippage, position sizing, risk management, regime changes, or other real-world trading constraints.

## 1. Setup

Import analysis and modeling libraries, configure project-relative paths, and create the figures directory. Visualizations use **matplotlib only**, with one standalone figure per chart and no subplots.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from modeling import (  # noqa: E402
    DATE_COLUMN,
    RANDOM_STATE,
    TARGET_COLUMN,
    TRAIN_DATE_FRACTION,
    chronological_train_test_split,
    get_feature_columns,
    load_modeling_data,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "stock_features.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)

## 2. Define the Target Variable

The target variable is `target_next_day_up`.

For each stock and trading date:

- `next_day_return = (next_trading_day_close / current_close) - 1`
- `target_next_day_up = 1` when `next_day_return > 0`
- `target_next_day_up = 0` when `next_day_return <= 0`

This means the model uses information available on the current row to predict whether the next trading day's return is positive.

In [ ]:
feature_columns = get_feature_columns()
print(f"Target column: {TARGET_COLUMN}")
print(f"Number of model features: {len(feature_columns)}")
feature_columns

## 3. Load and Prepare the Modeling Dataset

Load `data/processed/stock_features.csv` with the same preparation logic used in `src/modeling.py`:

1. Confirm that the date, target, and selected feature columns exist.
2. Parse `date` as datetime.
3. Convert the target and feature columns to numeric values.
4. Replace infinite values with missing values.
5. Drop rows missing the date, target, or selected feature columns.
6. Cast `target_next_day_up` to integer labels.

The feature list is imported from `get_feature_columns()` so the notebook uses the same inputs as the reusable modeling script.

In [ ]:
modeling_data = load_modeling_data(DATA_PATH)
modeling_data = modeling_data.sort_values([DATE_COLUMN]).reset_index(drop=True)

print(f"Rows after modeling preparation: {modeling_data.shape[0]:,}")
print(f"Columns: {modeling_data.shape[1]:,}")
print(f"Date range: {modeling_data[DATE_COLUMN].min().date()} to {modeling_data[DATE_COLUMN].max().date()}")
print("Target distribution:")
display(modeling_data[TARGET_COLUMN].value_counts(normalize=True).rename("share").to_frame())

display(modeling_data[[DATE_COLUMN, TARGET_COLUMN, *feature_columns]].head())

## 4. Chronological Train-Test Split

A chronological split is used instead of a random split because financial time series are ordered observations. Random splitting can let the training set include future market conditions relative to rows in the test set, which creates an unrealistic evaluation setup and can overstate model performance.

The split below trains on the earliest 80% of trading dates and tests on the latest 20% of trading dates. This better resembles a real forecasting workflow: train on historical observations, then evaluate on later unseen dates.

In [ ]:
X_train, X_test, y_train, y_test = chronological_train_test_split(
    modeling_data,
    train_fraction=TRAIN_DATE_FRACTION,
)

train_dates = modeling_data.loc[X_train.index, DATE_COLUMN]
test_dates = modeling_data.loc[X_test.index, DATE_COLUMN]

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Testing rows: {X_test.shape[0]:,}")
print(f"Training date range: {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Testing date range: {test_dates.min().date()} to {test_dates.max().date()}")

## 5. Train Models

Train four classifiers:

- `DummyClassifier(strategy="most_frequent")` as a majority-class baseline.
- Logistic regression with standard scaling.
- Random forest.
- Gradient boosting.

In [ ]:
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            ),
        ]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained {model_name}")

## 6. Evaluate Models

Evaluate each model using accuracy, precision, recall, F1, ROC-AUC, and a confusion matrix.

ROC-AUC is used for the primary comparison chart because it evaluates the model's ability to rank positive cases above negative cases across classification thresholds.

In [ ]:
def positive_class_scores(model, X):
    # Return positive-class scores when the estimator supports them.
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


evaluation_rows = []
confusion_matrices = {}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_score = positive_class_scores(model, X_test)
    roc_auc = roc_auc_score(y_test, y_score) if y_score is not None and y_test.nunique() == 2 else np.nan

    evaluation_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
        }
    )
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred, labels=[0, 1])

model_comparison = (
    pd.DataFrame(evaluation_rows)
    .sort_values("roc_auc", ascending=False, na_position="last")
    .reset_index(drop=True)
)

best_model_name = model_comparison.loc[0, "model"]
best_model = models[best_model_name]

print(f"Best model by ROC-AUC: {best_model_name}")
display(model_comparison)

In [ ]:
for model_name, matrix in confusion_matrices.items():
    print(f"{model_name} confusion matrix")
    display(pd.DataFrame(matrix, index=["actual_0", "actual_1"], columns=["predicted_0", "predicted_1"]))

## 7. Save Model Comparison ROC-AUC Bar Chart

Save a standalone bar chart comparing models by ROC-AUC.

In [ ]:
roc_auc_chart_path = FIGURES_DIR / "model_comparison_roc_auc.png"
plot_data = model_comparison.dropna(subset=["roc_auc"]).sort_values("roc_auc", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_data["model"], plot_data["roc_auc"], color="#4C78A8")
plt.xlabel("ROC-AUC")
plt.ylabel("Model")
plt.title("Model Comparison by ROC-AUC")
plt.xlim(0, 1)
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(roc_auc_chart_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved chart to {roc_auc_chart_path}")

## 8. Save Confusion Matrix for the Best Model

Save a standalone confusion matrix chart for the best model selected by ROC-AUC.

In [ ]:
confusion_matrix_chart_path = FIGURES_DIR / "best_model_confusion_matrix.png"
best_matrix = confusion_matrices[best_model_name]

plt.figure(figsize=(6, 5))
plt.imshow(best_matrix, cmap="Blues")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.xticks([0, 1], ["Down/Flat (0)", "Up (1)"])
plt.yticks([0, 1], ["Down/Flat (0)", "Up (1)"])
plt.colorbar(label="Count")

for row_index in range(best_matrix.shape[0]):
    for column_index in range(best_matrix.shape[1]):
        plt.text(
            column_index,
            row_index,
            f"{best_matrix[row_index, column_index]:,}",
            ha="center",
            va="center",
            color="black",
        )

plt.tight_layout()
plt.savefig(confusion_matrix_chart_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved chart to {confusion_matrix_chart_path}")

## 9. Save Feature Importance Chart When Supported

If the best model exposes `feature_importances_`, save a standalone feature importance bar chart. Tree-based models such as random forest and gradient boosting support this attribute; dummy and logistic regression models do not expose it directly in this workflow.

In [ ]:
feature_importance_chart_path = FIGURES_DIR / "best_model_feature_importance.png"

if hasattr(best_model, "feature_importances_"):
    feature_importance = (
        pd.DataFrame(
            {
                "feature": feature_columns,
                "importance": best_model.feature_importances_,
            }
        )
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    display(feature_importance)

    top_importance = feature_importance.head(15).sort_values("importance", ascending=True)

    plt.figure(figsize=(9, 6))
    plt.barh(top_importance["feature"], top_importance["importance"], color="#F58518")
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.title(f"Top Feature Importances: {best_model_name}")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(feature_importance_chart_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved chart to {feature_importance_chart_path}")
else:
    print(f"{best_model_name} does not expose feature_importances_; no feature importance chart was saved.")

## Modeling Takeaways

- **Best model:** _Placeholder: summarize which model achieved the strongest ROC-AUC and whether it meaningfully improved over the dummy baseline._
- **Classification trade-offs:** _Placeholder: summarize the precision, recall, and F1 trade-offs for the best model._
- **Confusion matrix interpretation:** _Placeholder: describe the most common correct and incorrect predictions._
- **Feature importance:** _Placeholder: if available, summarize which historical price or volume features were most influential._
- **Limitations:** Historical price and volume features alone are not enough to guarantee profitable trading performance. _Placeholder: add notes about transaction costs, slippage, changing market regimes, and the need for out-of-sample validation before any practical use._